# 🦀 Day 6: Memory Layout — repr(C), Alignment, size_of
---

Today we explore **memory layout** — how Rust arranges data in memory.

- **`size_of`** and **`align_of`**: Inspect type size and alignment
- **Default Rust layout**: Compiler may reorder fields
- **`#[repr(C)]`**: C-compatible, predictable field order
- **`#[repr(transparent)]`**: Newtype with same layout as inner
- **`#[repr(packed)]`** and **`#[repr(align(N))]`**
- **Enums**: Discriminant + largest variant
- **ZSTs**: Zero-sized types like `()` and `PhantomData`

In [ ]:
// size_of and align_of
use std::mem;

println!("size_of i32: {} bytes", mem::size_of::<i32>());
println!("align_of i32: {} bytes", mem::align_of::<i32>());
println!("size_of (i32, u64): {} bytes", mem::size_of::<(i32, u64)>());
println!("size_of Option<i32>: {} bytes", mem::size_of::<Option<i32>>());
println!("size_of (): {} bytes (ZST)", mem::size_of::<()>());

In [ ]:
// #[repr(C)]: C-compatible layout, predictable field order
#[repr(C)]
struct PointC {
    x: i32,
    y: i32,
}

struct PointRust {
    x: i32,
    y: i32,
}

println!("repr(C) Point: size={}, align={}", mem::size_of::<PointC>(), mem::align_of::<PointC>());
println!("Default Point: size={}, align={}", mem::size_of::<PointRust>(), mem::align_of::<PointRust>());

// #[repr(transparent)]: newtype with same layout as inner
#[repr(transparent)]
struct Meters(f64);
println!("Meters size = f64 size: {}", mem::size_of::<Meters>() == mem::size_of::<f64>());

In [ ]:
// Padding and alignment: larger types need alignment
#[repr(C)]
struct WithPadding {
    a: u8,
    b: u32,
}

// a: 1 byte, 3 bytes padding, b: 4 bytes = 8 total
println!("WithPadding size: {} (a=1 + padding + b=4)", mem::size_of::<WithPadding>());

// Enums: discriminant + largest variant
enum Simple {
    A,
    B(i32),
    C { x: u64 },
}
println!("Simple enum size: {} (discriminant + u64)", mem::size_of::<Simple>());

In [ ]:
// Zero-sized types (ZSTs)
use std::marker::PhantomData;

println!("(): {} bytes", mem::size_of::<()>());
println!("PhantomData<i32>: {} bytes", mem::size_of::<PhantomData<i32>>());

// transmute: reinterpret bytes (DANGEROUS — use sparingly!)
let x: u32 = 0x12345678;
let bytes: [u8; 4] = unsafe { mem::transmute(x) };
println!("u32 as bytes: {:02x?}", bytes);

## 📝 Exercise: Predict and verify sizes/alignments

Predict the `size_of` and `align_of` for:
- `struct S { a: u8, b: u16, c: u32 }` with `#[repr(C)]`
- `struct S { a: u32, b: u8 }` with `#[repr(C)]`

Then verify with `mem::size_of` and `mem::align_of`.

In [ ]:
// YOUR CODE HERE

## 🎯 Key Takeaways

1. **`size_of::<T>()`** and **`align_of::<T>()`** inspect layout
2. **`#[repr(C)]`** gives C-compatible, predictable field order
3. **`#[repr(transparent)]`** for newtypes with identical layout
4. **Padding** aligns fields; order affects total size
5. **Enums** = discriminant + size of largest variant
6. **ZSTs** (`()`, `PhantomData`) have size 0
7. **`transmute`** is dangerous — avoid unless necessary

---
**Tomorrow:** Mini-project — safe wrapper around a C library